In [ ]:
import os
import cv2
import numpy as np
import time

WIDTH, HEIGHT = 1100, 800

script_dir = os.path.dirname(os.path.abspath(os.getcwd()))

def Webcam(cameras : int):
    if type(cameras) == int and cameras:
        return cv2.VideoCapture(cameras)
    
camera = Webcam(0)

if not camera.isOpened():
    raise RuntimeError("Failed to Access Webcam")

cv2.namedWindow("Virtual Keyboard", cv2.WINDOW_NORMAL)

sentence = ""

BUTTON_WIDTH = 72
BUTTON_HEIGHT = 58
BUTTON_GAP = 8

DWELL_TIME = 0.45

hovered_key = None
hover_started = 0
already_activated = False

FRAME_AREA_THRESHOLD = 4000

keyboard_rows = [
    ("QWERTYUIOP", 150, 250),
    ("ASDFGHJKL", 190, 318),
    ("ZXCVBNM", 270, 386)
]

buttons = []

for row, start_x, start_y in keyboard_rows:

    for index, key in enumerate(row):

        x1 = start_x + index * (
            BUTTON_WIDTH + BUTTON_GAP
        )

        y1 = start_y

        buttons.append(
            (
                key,
                x1,
                y1,
                x1 + BUTTON_WIDTH,
                y1 + BUTTON_HEIGHT
            )
        )

control_y = 454
control_height = 58

controls = [
    ("SPACE", 175, 300),
    ("BACKSPACE", 485, 180),
    ("CLEAR", 675, 120),
    ("ENTER", 805, 120)
]

for key, x, width in controls:

    buttons.append(
        (
            key,
            x,
            control_y,
            x + width,
            control_y + control_height
        )
    )


def activate_key(key):

    global sentence

    if key == "SPACE":

        if sentence and not sentence.endswith(" "):
            sentence += " "

    elif key == "BACKSPACE":

        sentence = sentence[:-1]

    elif key == "CLEAR":

        sentence = ""
    elif key == "ENTER":
        sentence += "\n"
    else:
        sentence += key.lower()


def draw_sentence(frame):

    x1, y1 = 90, 585
    x2, y2 = 1010, 775

    cv2.rectangle(
        frame,
        (x1, y1),
        (x2, y2),
        (35, 35, 35),
        -1
    )

    cv2.rectangle(
        frame,
        (x1, y1),
        (x2, y2),
        (255, 255, 255),
        2
    )

    cv2.putText(
        frame,
        "YOUR SENTENCE",
        (x1 + 20, y1 + 32),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (0, 200, 255),
        2
    )

    lines = [""]

    for character in sentence:

        if character == "\n":

            lines.append("")
            continue

        candidate = lines[-1] + character

        text_width = cv2.getTextSize(
            candidate,
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            2
        )[0][0]

        if text_width < x2 - x1 - 40:

            lines[-1] = candidate

        else:

            lines.append(character)

    visible_lines = lines[-4:]

    for index, line in enumerate(visible_lines):

        cv2.putText(
            frame,
            line,
            (x1 + 20, y1 + 75 + index * 32),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (255, 255, 255),
            2
        )


while True:

    success, frame = camera.read()

    if not success:
        break

    frame = cv2.flip(frame, 1)

    frame = cv2.resize(
        frame,
        (WIDTH, HEIGHT)
    )

    hsv = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2HSV
    )

    ycrcb = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2YCrCb
    )

    hsv_mask = cv2.inRange(
        hsv,
        np.array([0, 30, 60]),
        np.array([30, 200, 255])
    )

    ycrcb_mask = cv2.inRange(
        ycrcb,
        np.array([0, 125, 65]),
        np.array([255, 180, 140])
    )

    mask = cv2.bitwise_and(
        hsv_mask,
        ycrcb_mask
    )

    kernel = np.ones((5, 5), np.uint8)

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_OPEN,
        kernel
    )

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_CLOSE,
        kernel
    )

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    fingertip = None

    if contours:

        hand = max(
            contours,
            key=cv2.contourArea
        )

        if cv2.contourArea(hand) > FRAME_AREA_THRESHOLD:

            fingertip = tuple(
                hand[hand[:, :, 1].argmin()][0]
            )

            cv2.drawContours(
                frame,
                [hand],
                -1,
                (0, 255, 0),
                2
            )

            cv2.circle(
                frame,
                fingertip,
                10,
                (0, 0, 255),
                -1
            )

    current_hover = None

    if fingertip is not None:

        finger_x, finger_y = fingertip

        for key, x1, y1, x2, y2 in buttons:

            if (
                x1 <= finger_x <= x2
                and y1 <= finger_y <= y2
            ):

                current_hover = key
                break

    current_time = time.monotonic()

    if current_hover != hovered_key:

        hovered_key = current_hover
        hover_started = current_time
        already_activated = False

    elif (
        hovered_key is not None
        and not already_activated
        and current_time - hover_started >= DWELL_TIME
    ):

        activate_key(hovered_key)
        already_activated = True


    overlay = frame.copy()

    cv2.rectangle(
        overlay,
        (120, 215),
        (980, 535),
        (25, 25, 25),
        -1
    )

    frame = cv2.addWeighted(
        overlay,
        0.88,
        frame,
        0.12,
        0
    )

    cv2.putText(
        frame,
        "Virtual Keyboard",
        (390, 195),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.9,
        (255, 255, 255),
        2
    )


    for key, x1, y1, x2, y2 in buttons:

        if key == hovered_key:

            color = (50, 190, 80)

        elif key in ["SPACE", "ENTER"]:

            color = (180, 110, 40)

        elif key in ["BACKSPACE", "CLEAR"]:

            color = (70, 70, 180)

        else:

            color = (75, 75, 75)

        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            color,
            -1
        )

        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            (255, 255, 255),
            2
        )

        font_scale = 0.55 if len(key) > 1 else 0.8

        text_size = cv2.getTextSize(
            key,
            cv2.FONT_HERSHEY_SIMPLEX,
            font_scale,
            2
        )[0]

        text_x = x1 + (
            x2 - x1 - text_size[0]
        ) // 2

        text_y = y1 + (
            y2 - y1 + text_size[1]
        ) // 2

        cv2.putText(
            frame,
            key,
            (text_x, text_y),
            cv2.FONT_HERSHEY_SIMPLEX,
            font_scale,
            (255, 255, 255),
            2
        )

    draw_sentence(frame)

    cv2.putText(
        frame,
        "Hold your fingertip over a key to type | Q to quit",
        (240, 570),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (255, 255, 255),
        1
    )

    cv2.imshow(
        "Virtual Keyboard",
        frame
    )

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

camera.release()
cv2.destroyAllWindows()